In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "src").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_transactions

DATASET_PATH = PROJECT_ROOT / "data" / "raw" / "online_retail_II.xlsx"
df = load_transactions(DATASET_PATH)
df.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


## Dataset overview

The dataset is loaded through the project loader, which reads both Excel sheets and normalizes the source columns. The first checks describe the dataset size and data types.

In [6]:
overview = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "unique_invoices",
            "unique_products",
            "unique_customers",
            "date_from",
            "date_to",
        ],
        "value": [
            len(df),
            df.shape[1],
            df["invoice_no"].nunique(),
            df["stock_code"].nunique(),
            df["customer_id"].nunique(),
            df["invoice_date"].min(),
            df["invoice_date"].max(),
        ],
    }
)
overview

,metric,value
0,rows,1067371
1,columns,8
2,unique_invoices,53628
3,unique_products,5305
4,unique_customers,5942
5,date_from,2009-12-01 07:45:00
6,date_to,2011-12-09 12:50:00


## Data quality checks

The Online Retail II dataset contains canceled invoices and transactions without a customer identifier. These records are not removed in the exploratory stage; instead, they are measured explicitly before preprocessing.

In [7]:
quality = pd.DataFrame(
    {
        "metric": [
            "missing_customer_id",
            "missing_description",
            "duplicate_rows",
            "negative_quantity_rows",
            "zero_price_rows",
        ],
        "count": [
            df["customer_id"].isna().sum(),
            df["description"].isna().sum(),
            df.duplicated().sum(),
            (df["quantity"] < 0).sum(),
            (df["unit_price"] == 0).sum(),
        ],
    }
)
quality["share_percent"] = (quality["count"] / len(df) * 100).round(2)
quality

,metric,count,share_percent
0,missing_customer_id,243007,22.77
1,missing_description,4382,0.41
2,duplicate_rows,34335,3.22
3,negative_quantity_rows,22950,2.15
4,zero_price_rows,6202,0.58


## Sales analysis

Revenue is calculated at line-item level as `quantity * unit_price`. For the first business view, sales and returns are shown separately.

In [8]:
analysis_df = df.copy()
analysis_df["revenue"] = analysis_df["quantity"] * analysis_df["unit_price"]
analysis_df["month"] = analysis_df["invoice_date"].dt.to_period("M").astype(str)
analysis_df["is_return"] = analysis_df["quantity"] < 0

sales_summary = pd.DataFrame(
    {
        "metric": [
            "gross_revenue",
            "return_value",
            "net_revenue",
            "sales_lines",
            "return_lines",
            "average_line_value",
        ],
        "value": [
            analysis_df.loc[~analysis_df["is_return"], "revenue"].sum(),
            analysis_df.loc[analysis_df["is_return"], "revenue"].sum(),
            analysis_df["revenue"].sum(),
            (~analysis_df["is_return"]).sum(),
            analysis_df["is_return"].sum(),
            analysis_df["revenue"].mean(),
        ],
    }
)
sales_summary

,metric,value
0,gross_revenue,2.081429e+07
1,return_value,-1.527041e+06
2,net_revenue,1.928725e+07
3,sales_lines,1.044421e+06
4,return_lines,2.295000e+04
5,average_line_value,1.806987e+01


In [9]:
monthly_sales = (
    analysis_df.loc[~analysis_df["is_return"]]
    .groupby("month", as_index=False)
    .agg(
        revenue=("revenue", "sum"),
        orders=("invoice_no", "nunique"),
        customers=("customer_id", "nunique"),
    )
)
monthly_sales["aov"] = (monthly_sales["revenue"] / monthly_sales["orders"]).round(2)
monthly_sales.tail(12)

,month,revenue,orders,customers,aov
13,2011-01,691364.560,1120,741,617.29
14,2011-02,523631.890,1126,758,465.04
15,2011-03,717639.360,1531,974,468.74
16,2011-04,537808.621,1318,856,408.05
17,2011-05,770536.020,1731,1056,445.14
18,2011-06,761739.900,1576,991,483.34
19,2011-07,719221.191,1540,949,467.03
20,2011-08,737014.260,1409,935,523.08
21,2011-09,1058590.172,1896,1266,558.33
22,2011-10,1154979.300,2129,1364,542.50


## Geographic and product concentration

These tables identify the countries and products that contribute most to gross sales. They will be useful inputs for the later customer and retention analysis.

In [10]:
top_countries = (
    analysis_df.loc[~analysis_df["is_return"]]
    .groupby("country", as_index=False)
    .agg(revenue=("revenue", "sum"), orders=("invoice_no", "nunique"))
    .sort_values("revenue", ascending=False)
    .head(10)
)
top_countries["revenue_share_percent"] = (
    top_countries["revenue"] / top_countries["revenue"].sum() * 100
).round(2)
top_countries

,country,revenue,orders,revenue_share_percent
40,United Kingdom,1.771268e+07,38401,87.42
11,EIRE,6.644318e+05,626,3.28
26,Netherlands,5.542323e+05,229,2.74
15,Germany,4.312625e+05,789,2.13
14,France,3.569446e+05,622,1.76
0,Australia,1.699681e+05,95,0.84
34,Spain,1.091785e+05,154,0.54
36,Switzerland,1.010113e+05,93,0.50
35,Sweden,9.190372e+04,105,0.45
10,Denmark,6.986219e+04,43,0.34


In [11]:
top_products = (
    analysis_df.loc[~analysis_df["is_return"]]
    .groupby(["stock_code", "description"], dropna=False, as_index=False)
    .agg(revenue=("revenue", "sum"), units_sold=("quantity", "sum"))
    .sort_values("revenue", ascending=False)
    .head(10)
)
top_products

,stock_code,description,revenue,units_sold
2352,22423,REGENCY CAKESTAND 3 TIER,344563.25,27594
6864,M,Manual,341089.85,10058
6862,DOT,DOTCOM POSTAGE,322657.48,1441
6146,85123A,WHITE HANGING HEART T-LIGHT HOLDER,262931.16,96091
3978,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995
4475,47566,PARTY BUNTING,149187.05,28395
6106,85099B,JUMBO BAG RED RETROSPOT,148823.92,78866
5686,84879,ASSORTED COLOUR BIRD ORNAMENT,132187.92,81817
6866,POST,POSTAGE,127597.42,5461
1874,22086,PAPER CHAIN KIT 50'S CHRISTMAS,123141.54,36581


## Initial findings

The analysis establishes the baseline for the next stages:

- returns must remain identifiable during preprocessing;
- missing `customer_id` values limit customer-level and churn analysis;
- revenue should be calculated from quantity and unit price rather than treated as a raw column;
- monthly sales, country concentration, and product concentration are the first business views;
- the next implementation stage is data validation and preprocessing rules.